In [0]:
%sh
pip install gspread pymongo[srv]

In [0]:
%sh pwd

In [0]:
import sys
sys.path.append("/Workspace/Users/matumazparrote@gmail.com/elt_products_scraping")

In [0]:
import json
from google.oauth2 import service_account
import os
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
from src.config.mongo_db import mongo_db
from src.utils.google_sheet import google_sheets_handler
from datetime import datetime

creds_json = dbutils.secrets.get(scope="gcp-creds", key="google-credentials")
creds_dict = json.loads(creds_json)
print("Credentials loaded successfully")
GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS products")

In [0]:
product_catalog = google_sheets_handler.read(creds_dict, GOOGLE_SCOPES)
product_catalog_spark = spark.createDataFrame(product_catalog)
product_catalog_spark.write.mode("overwrite").saveAsTable("products.catalog")

In [0]:
# spark.sql("SHOW TABLES IN products").show()

In [0]:
base_volume = "/Volumes/workspace/products/products_tracker"

In [0]:
mongo_db.connect(dbutils.secrets.get("mondo_db_creds", "MONGO_DB_URI"), db_name="products")

In [0]:
scraped_products_collection = mongo_db.get_database()["scraped_products"]
current_timestamp = datetime.now().strftime("%Y-%m-%d")[0:10]
print(f"Timestamp actual: {current_timestamp}")

In [0]:
from bson import json_util
scraped_data = list(scraped_products_collection.find({
    "scraped_at": {"$regex": current_timestamp}
}))
print(f"Cantidad: {len(scraped_data)}")
year, month, day = current_timestamp.split("-")
path = f"{base_volume}/scraped/year={year}/month={month}/day={day}"
dbutils.fs.mkdirs(path)
full_path = f"{path}/{current_timestamp}.json"
with open(full_path, "w", encoding="utf-8") as f:
    f.write(json_util.dumps(scraped_data, indent=4))
print(f"✅ Guardado: {full_path}")